# Silver Evaluation (Intersection) — Raw vs. Quality-Aware

This notebook evaluates the ATE output of three methods (RB, UNSUP, HYB) on the same segment intersection (intersection of segment IDs), under two scenarios:

1. Raw silver (agreement): pseudo-label per segment = overlap(RB, UNSUP) without context filtering.
2. Quality-aware silver: candidates are ranked using context-fit (cosine similarity term–segment embedding)

In [ ]:
##(Optional) Mount Google Drive if running on Google Colab
# try:
#     from google.colab import drive  # type: ignore
#     drive.mount('/content/drive')
#     IN_COLAB = True
# except Exception:
#     IN_COLAB = False
# print("IN_COLAB:", IN_COLAB)


# 1. Setup

In [ ]:
# ==== Setup ====
import os, re, ast, json, pickle
import numpy as np
import pandas as pd


# Base folder (change according to your data location)
# - Colab: "/content/drive/MyDrive/dataset_baseline"
# - Lokal: gunakan path lokal atau relative folder
BASE_DIR = "./data/dataset_baseline"

# Output folder
OUT_DIR = os.path.join("./outputs", "silver_intersection")
os.makedirs(OUT_DIR, exist_ok=True)
print("BASE_DIR:", BASE_DIR)
print("OUT_DIR :", OUT_DIR)

# Helper: safe parse list from string
def parse_list_cell(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]
    s = str(x).strip()
    if not s or s.lower() in {"nan","none","[]","kosong"}:
        return []
    # try JSON / python-literal
    try:
        v = json.loads(s)
        if isinstance(v, list):
            return [str(t).strip() for t in v if str(t).strip()]
    except Exception:
        pass
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [str(t).strip() for t in v if str(t).strip()]
    except Exception:
        pass
    # fallback: split by comma
    return [t.strip() for t in re.split(r"[;,]", s) if t.strip()]

def uniq_sorted(xs):
    seen=set()
    out=[]
    for x in xs:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# 2. Load Data

In [ ]:
# ==== Load Data ====
import glob

csvs = sorted(glob.glob(os.path.join(BASE_DIR, "*.csv")))
print("CSV files in BASE_DIR:")
for p in csvs[:50]:
    print(" -", os.path.basename(p))
if len(csvs) > 50:
    print(f"... ({len(csvs)-50} more)")

# Set manual path (adjust file name if different)
PATH_RB  = os.path.join(BASE_DIR, "rb_apect_terms_full_with_segtext.csv")
PATH_UN  = os.path.join(BASE_DIR, "unsup_terms_ranked_full_with_segtext.csv")
PATH_HYB = os.path.join(BASE_DIR, "dataset_hybrid_best_terms_sbert.csv")

for label, p in [("RB", PATH_RB), ("UNSUP", PATH_UN), ("HYB", PATH_HYB)]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{label} file tidak ditemukan: {p}")

print("PATH_RB :", PATH_RB)
print("PATH_UN :", PATH_UN)
print("PATH_HYB:", PATH_HYB)

rb  = pd.read_csv(PATH_RB)
uns = pd.read_csv(PATH_UN)
hyb = pd.read_csv(PATH_HYB)

print("\nShapes:")
print("RB   :", rb.shape)
print("UNSUP:", uns.shape)
print("HYB  :", hyb.shape)


# 3. Standardize key & term columns

In [ ]:
# ==== Standardize key & term columns ====
KEY_COLS = ["comment_id", "seg_id"]

def ensure_cols(df, name):
    miss = [c for c in KEY_COLS if c not in df.columns]
    if miss:
        raise ValueError(f"{name} missing key columns: {miss}")

ensure_cols(rb, "RB")
ensure_cols(uns, "UNSUP")
ensure_cols(hyb, "HYB")

# pastikan tipe key konsisten (int nullable)
for df in [rb, uns, hyb]:
    for k in KEY_COLS:
        df[k] = pd.to_numeric(df[k], errors="coerce").astype("Int64")


# kolom term (sesuaikan jika nama berbeda)
RB_TERMS_COL = "aspect_terms_rb_clean"
UN_TERMS_COL = "aspect_terms_unsup"
HYB_TERMS_COL = "hybrid_topk_terms" if "hybrid_topk_terms" in hyb.columns else ("hybrid_topk3_terms" if "hybrid_topk3_terms" in hyb.columns else None)

if RB_TERMS_COL not in rb.columns:
    raise ValueError(f"RB_TERMS_COL '{RB_TERMS_COL}' tidak ditemukan pada RB")
if UN_TERMS_COL not in uns.columns:
    raise ValueError(f"UN_TERMS_COL '{UN_TERMS_COL}' tidak ditemukan pada UNSUP")
if HYB_TERMS_COL is None:
    raise ValueError("Kolom hybrid terms tidak ditemukan (cari 'hybrid_topk_terms' / 'hybrid_topk3_terms')")

# kolom teks segmen
TEXT_COL_CANDIDATES = ["seg_text", "seg_text_raw", "seg_text_norm", "seg_text_clean"]
def pick_text_col(df):
    for c in TEXT_COL_CANDIDATES:
        if c in df.columns:
            return c
    return None

RB_TEXT = pick_text_col(rb)
UN_TEXT = pick_text_col(uns)
HYB_TEXT = pick_text_col(hyb)

# gunakan teks dari HYB jika tersedia, fallback RB, lalu UNSUP
print("Text col RB  :", RB_TEXT)
print("Text col UN  :", UN_TEXT)
print("Text col HYB :", HYB_TEXT)


# 4. Intersection of segment IDs

In [ ]:
# ==== Intersection of segment IDs ====
def key_df(df):
    return df[KEY_COLS].dropna().drop_duplicates()

keys = key_df(rb).merge(key_df(uns), on=KEY_COLS, how="inner").merge(key_df(hyb), on=KEY_COLS, how="inner")
print("N RB keys      :", len(key_df(rb)))
print("N UNSUP keys   :", len(key_df(uns)))
print("N HYB keys     :", len(key_df(hyb)))
print("N intersection :", len(keys))

rb_i  = keys.merge(rb,  on=KEY_COLS, how="left", validate="one_to_one")
uns_i = keys.merge(uns, on=KEY_COLS, how="left", validate="one_to_one")
hyb_i = keys.merge(hyb, on=KEY_COLS, how="left", validate="one_to_one")

# build base (audit-friendly)
base = keys.copy()

# seg_text: prioritas HYB lalu RB lalu UNSUP
def coalesce_text(row):
    for src, col in [("hyb", HYB_TEXT), ("rb", RB_TEXT), ("uns", UN_TEXT)]:
        if col is None:
            continue
        v = row.get(f"{src}_{col}", None)
        if v is not None and not (isinstance(v,float) and np.isnan(v)) and str(v).strip():
            return str(v)
    return ""

# attach text cols with prefixes
for df, pref, tcol in [(rb_i,"rb",RB_TEXT),(uns_i,"uns",UN_TEXT),(hyb_i,"hyb",HYB_TEXT)]:
    if tcol is not None:
        base = base.merge(df[KEY_COLS+[tcol]].rename(columns={tcol:f"{pref}_{tcol}"}), on=KEY_COLS, how="left")

base["seg_text"] = base.apply(coalesce_text, axis=1)

# attach term lists
base = base.merge(rb_i[KEY_COLS+[RB_TERMS_COL]].rename(columns={RB_TERMS_COL:"rb_terms_raw"}), on=KEY_COLS, how="left")
base = base.merge(uns_i[KEY_COLS+[UN_TERMS_COL]].rename(columns={UN_TERMS_COL:"unsup_terms_raw"}), on=KEY_COLS, how="left")
base = base.merge(hyb_i[KEY_COLS+[HYB_TERMS_COL]].rename(columns={HYB_TERMS_COL:"hyb_terms_raw"}), on=KEY_COLS, how="left")

base["rb_terms"] = base["rb_terms_raw"].apply(lambda x: uniq_sorted(parse_list_cell(x)))
base["unsup_terms"] = base["unsup_terms_raw"].apply(lambda x: uniq_sorted(parse_list_cell(x)))
base["hyb_terms"] = base["hyb_terms_raw"].apply(lambda x: uniq_sorted(parse_list_cell(x)))

display(base[[*KEY_COLS,"seg_text","rb_terms","unsup_terms","hyb_terms"]].head(3))


# 5. Silver by agreement (Raw)

In [ ]:
# ==== Silver by agreement (Raw) ====
# pseudo-label per segment = overlap(RB, UNSUP) without context filtering

base["silver_terms_raw"] = base.apply(lambda r: sorted(list(set(r["rb_terms"]) & set(r["unsup_terms"]))), axis=1)
base["n_silver_raw"] = base["silver_terms_raw"].apply(len)

print("Segmen dengan silver_raw > 0:", int((base["n_silver_raw"]>0).sum()), "/", len(base))
display(base[[*KEY_COLS,"silver_terms_raw","n_silver_raw"]].head(5))

def pr_silver(pred_terms, silver_terms):
    P, S = set(pred_terms), set(silver_terms)
    # aturan empty-case agar tidak bias NaN-drop
    if not S and not P:
        return 1.0, 1.0
    if not S and P:
        return 0.0, 0.0
    if S and not P:
        return 0.0, 0.0
    tp = len(P & S)
    prec = tp / len(P) if len(P)>0 else 0.0
    rec  = tp / len(S) if len(S)>0 else 0.0
    return prec, rec

for m, col in [("rule_based","rb_terms"),("unsupervised","unsup_terms"),("hybrid_topk","hyb_terms")]:
    base[f"{m}_prec_raw"], base[f"{m}_rec_raw"] = zip(*base.apply(lambda r: pr_silver(r[col], r["silver_terms_raw"]), axis=1))

summary_raw = pd.DataFrame([
    {"method":"rule_based", "silver_prec_mean": base["rule_based_prec_raw"].mean(), "silver_recall_mean": base["rule_based_rec_raw"].mean()},
    {"method":"unsupervised","silver_prec_mean": base["unsupervised_prec_raw"].mean(), "silver_recall_mean": base["unsupervised_rec_raw"].mean()},
    {"method":"hybrid_topk","silver_prec_mean": base["hybrid_topk_prec_raw"].mean(), "silver_recall_mean": base["hybrid_topk_rec_raw"].mean()},
])

display(summary_raw)

summary_raw.to_csv(os.path.join(OUT_DIR, "summary_silver_raw_intersection.csv"), index=False)
base.to_csv(os.path.join(OUT_DIR, "master_silver_raw_intersection.csv"), index=False)
print("Saved:", "summary_silver_raw_intersection.csv", "master_silver_raw_intersection.csv")


# 6. Generate Embeddings for Context-fit

In [ ]:
# ==== Context-Fit embeddings (cosine(term_emb, seg_emb)) ====
# Menggunakan SentenceTransformer distiluse-base-multilingual-cased-v2 (multilingual)

try:
    import torch
    from sentence_transformers import SentenceTransformer
except Exception:
    !pip -q install -U sentence-transformers
    import torch
    from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

MODEL_NAME = "sentence-transformers/distiluse-base-multilingual-cased-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = SentenceTransformer(MODEL_NAME, device=device)

# segment embeddings cache
seg_emb_path = os.path.join(OUT_DIR, "seg_emb_intersection.npy")
seg_texts = base["seg_text"].fillna("").astype(str).tolist()

if os.path.exists(seg_emb_path):
    seg_emb = np.load(seg_emb_path)
    print("Loaded seg_emb:", seg_emb.shape)
else:
    seg_emb = model.encode(seg_texts, batch_size=64, show_progress_bar=True,
                           convert_to_numpy=True, normalize_embeddings=True)
    np.save(seg_emb_path, seg_emb)
    print("Saved seg_emb:", seg_emb.shape)

# term embedding cache (global unique terms)
all_terms = set()
for col in ["rb_terms","unsup_terms","hyb_terms"]:
    base[col].apply(lambda lst: [all_terms.add(t) for t in lst])
all_terms = sorted(list(all_terms))
print("Unique terms:", len(all_terms))

term_emb_path = os.path.join(OUT_DIR, "term_emb_intersection.pkl")
if os.path.exists(term_emb_path):
    with open(term_emb_path, "rb") as f:
        term2emb = pickle.load(f)
    print("Loaded term2emb:", len(term2emb))
else:
    term_vecs = model.encode(all_terms, batch_size=256, show_progress_bar=True,
                             convert_to_numpy=True, normalize_embeddings=True)
    term2emb = {t: term_vecs[i] for i,t in enumerate(all_terms)}
    with open(term_emb_path, "wb") as f:
        pickle.dump(term2emb, f)
    print("Saved term2emb:", len(term2emb))

# helper: score list of terms against a segment vector
def rank_terms_by_context(term_list, seg_vec, term2emb):
    if not term_list:
        return [], []
    vecs = [term2emb.get(t) for t in term_list if t in term2emb]
    terms = [t for t in term_list if t in term2emb]
    if not terms:
        return [], []
    sims = cosine_similarity(np.vstack(vecs), seg_vec.reshape(1,-1)).reshape(-1)
    order = np.argsort(-sims)
    terms_sorted = [terms[i] for i in order]
    sims_sorted  = [float(sims[i]) for i in order]
    return terms_sorted, sims_sorted


# 7. Silver Quality-Aware dengan Threshold Context-Fit

This section examines candidate filtering based on the cosine similarity threshold.

Reading sensitivity to $τ$.


In [ ]:
CONTEXT_FIT_THRESHOLDS = [0.5, 0.6, 0.7] # Define the context-fit thresholds

def filter_terms_by_threshold(term_list, seg_vec, term2emb, threshold):
    if not term_list or seg_vec is None or not term2emb:
        return []
    # Use rank_terms_by_context to get terms and their scores
    terms_sorted, sims_sorted = rank_terms_by_context(term_list, seg_vec, term2emb)
    # Filter terms where similarity score is above the threshold
    filtered_terms = [term for term, sim in zip(terms_sorted, sims_sorted) if sim >= threshold]
    return filtered_terms

# Prepare lists to store filtered terms for each threshold
for threshold in CONTEXT_FIT_THRESHOLDS:
    rb_filtered_col = f"rb_terms_cf_thresh_{str(threshold).replace('.', '_')}"
    unsup_filtered_col = f"unsup_terms_cf_thresh_{str(threshold).replace('.', '_')}"

    rb_filtered_terms_list = []
    unsup_filtered_terms_list = []

    print(f"Filtering terms for threshold: {threshold}")
    for i, row in base.iterrows():
        seg_vec = seg_emb[i] # Get the segment embedding for the current row

        # Filter RB terms
        rb_filtered = filter_terms_by_threshold(row["rb_terms"], seg_vec, term2emb, threshold)
        rb_filtered_terms_list.append(rb_filtered)

        # Filter UNSUP terms
        unsup_filtered = filter_terms_by_threshold(row["unsup_terms"], seg_vec, term2emb, threshold)
        unsup_filtered_terms_list.append(unsup_filtered)

    base[rb_filtered_col] = rb_filtered_terms_list
    base[unsup_filtered_col] = unsup_filtered_terms_list

print("Filtering completed for all thresholds.")
display(base[[*KEY_COLS, "rb_terms", "unsup_terms", "rb_terms_cf_thresh_0_5", "unsup_terms_cf_thresh_0_5"]].head())

for threshold in CONTEXT_FIT_THRESHOLDS:
    rb_filtered_col = f"rb_terms_cf_thresh_{str(threshold).replace('.', '_')}"
    unsup_filtered_col = f"unsup_terms_cf_thresh_{str(threshold).replace('.', '_')}"
    silver_cf_col = f"silver_terms_cf_thresh_{str(threshold).replace('.', '_')}"

    print(f"Generating context-aware silver terms for threshold: {threshold}")
    base[silver_cf_col] = base.apply(
        lambda r: sorted(list(set(r[rb_filtered_col]) & set(r[unsup_filtered_col]))),
        axis=1
    )
    base[f"n_silver_cf_thresh_{str(threshold).replace('.', '_')}"] = base[silver_cf_col].apply(len)

print("Context-aware silver terms generation completed for all thresholds.")
display(base[[*KEY_COLS, "silver_terms_raw", "silver_terms_cf_thresh_0_5"]].head())

rows_cf_eval = []

for threshold in CONTEXT_FIT_THRESHOLDS:
    silver_cf_col = f"silver_terms_cf_thresh_{str(threshold).replace('.', '_')}"

    # Filter predictions for each method based on the current threshold
    rb_pred_cf_col = f"rb_terms_cf_pred_thresh_{str(threshold).replace('.', '_')}"
    unsup_pred_cf_col = f"unsup_terms_cf_pred_thresh_{str(threshold).replace('.', '_')}"
    hyb_pred_cf_col = f"hyb_terms_cf_pred_thresh_{str(threshold).replace('.', '_')}"

    print(f"Filtering predictions and evaluating for threshold: {threshold}")
    rb_preds_filtered = []
    unsup_preds_filtered = []
    hyb_preds_filtered = []

    for i, row in base.iterrows():
        seg_vec = seg_emb[i]
        rb_preds_filtered.append(filter_terms_by_threshold(row["rb_terms"], seg_vec, term2emb, threshold))
        unsup_preds_filtered.append(filter_terms_by_threshold(row["unsup_terms"], seg_vec, term2emb, threshold))
        hyb_preds_filtered.append(filter_terms_by_threshold(row["hyb_terms"], seg_vec, term2emb, threshold))

    base[rb_pred_cf_col] = rb_preds_filtered
    base[unsup_pred_cf_col] = unsup_preds_filtered
    base[hyb_pred_cf_col] = hyb_preds_filtered

    # Evaluate against context-aware silver terms
    for m, pred_col in [
        ("rule_based", rb_pred_cf_col),
        ("unsupervised", unsup_pred_cf_col),
        ("hybrid_topk", hyb_pred_cf_col)
    ]:
        precs, recs = zip(*base.apply(lambda r: pr_silver(r[pred_col], r[silver_cf_col]), axis=1))
        rows_cf_eval.append({
            "method": m,
            "threshold": threshold,
            "silver_cf_prec_mean": float(np.mean(precs)),
            "silver_cf_recall_mean": float(np.mean(recs)),
        })

summary_cf_aware_silver = pd.DataFrame(rows_cf_eval).sort_values(["method","threshold"])
display(summary_cf_aware_silver)

summary_cf_aware_silver.to_csv(os.path.join(OUT_DIR, "summary_silver_contextaware_silver_intersection.csv"), index=False)
print("Saved: summary_silver_contextaware_silver_intersection.csv")

display(summary_cf_aware_silver)


## Note

- If running outside of Colab, ensure the `BASE_DIR` folder contains the three CSV files specified in the *Load Data* section.
- All table summaries (and some intermediate artifacts) are saved to `OUT_DIR` to maintain a clean audit trail.
